# TRACK 1 - DATA ENGINEERING

STEP 1 - Load & Inspect Raw Data

In [32]:
import pandas as pd

# Update paths with actual file locations
indices_path = "Data/Indices global 01-2005 à 10-2025.csv"
stocks_path = "Data/Stocks global 01-2005 à 10-2025.csv"

# Load files
indices_raw = pd.read_csv(indices_path, encoding='latin-1', low_memory=False)
stocks_raw = pd.read_csv(stocks_path, encoding='latin-1', low_memory=False)

print("=== INDICES COLUMNS ===")
print(indices_raw.columns.tolist())
print(indices_raw.head())

print("=== STOCKS COLUMNS ===")
print(stocks_raw.columns.tolist())
print(stocks_raw.head())


=== INDICES COLUMNS ===
['gvkeyx', 'datadate', 'indextype', 'tic', 'indexid', 'indexcat', 'idxiddesc', 'prccm']
   gvkeyx    datadate  indextype       tic indexid indexcat  \
0  115114  2005-01-31  COMPOSITE  I2JPN017  JASDAQ    EXCHG   
1  115114  2005-02-28  COMPOSITE  I2JPN017  JASDAQ    EXCHG   
2  115114  2005-03-31  COMPOSITE  I2JPN017  JASDAQ    EXCHG   
3  115114  2005-04-30  COMPOSITE  I2JPN017  JASDAQ    EXCHG   
4  115114  2005-05-31  COMPOSITE  I2JPN017  JASDAQ    EXCHG   

                            idxiddesc  prccm  
0  Japanese Over the Counter Exchange  95.69  
1  Japanese Over the Counter Exchange  95.49  
2  Japanese Over the Counter Exchange  95.68  
3  Japanese Over the Counter Exchange  95.58  
4  Japanese Over the Counter Exchange  93.24  
=== STOCKS COLUMNS ===
['fic', 'gvkey', 'datadate', 'conm', 'ajpm', 'prccm', 'curcddvm', 'dvpspm', 'exchg']
   fic  gvkey    datadate conm  ajpm     prccm curcddvm  dvpspm  exchg
0  NaN      5  2006-01-31  NaN   1.0  10864.86  

Step 3 - CLEANING PIPELINE

STEP 3A - Clean INDICES data

In [33]:
import pandas as pd
import numpy as np

COVERAGE_MIN = 0.80  # minimum share of available months
MIN_OBS = 60         # guardrail for extremely short histories
MIN_VOL = 1e-6       # eliminate near-constant price series


def build_return_matrix(
    raw_df,
    asset_cols,
    *,
    label,
    price_col="prccm",
    date_col="datadate",
    coverage_min=COVERAGE_MIN,
    min_obs=MIN_OBS,
    min_vol=MIN_VOL,
):
    """Transform a raw price panel into log-return matrices plus QA summary."""
    required_cols = list(asset_cols) + [date_col, price_col]
    missing_cols = [col for col in required_cols if col not in raw_df.columns]
    if missing_cols:
        raise KeyError(f"Missing columns for {label}: {missing_cols}")

    df = raw_df[required_cols].copy()
    df = df.rename(columns={date_col: "date", price_col: "price"})
    df = df.dropna(subset=list(asset_cols) + ["date", "price"])

    for col in asset_cols:
        df[col] = df[col].astype(str).str.strip()

    df["asset_id"] = df[asset_cols].agg("-".join, axis=1)
    df = df[["date", "asset_id", "price"]]

    df["date"] = pd.to_datetime(df["date"])
    df = df[df["price"] > 0]
    df = df.sort_values(["asset_id", "date"])

    # Aggregate duplicates prior to return calculation
    df = df.groupby(["asset_id", "date"], as_index=False)["price"].mean()

    price_span = df.groupby("asset_id")["price"].transform(lambda x: x.max() - x.min())
    df = df[price_span > 0]

    df["log_ret"] = df.groupby("asset_id")["price"].transform(lambda s: np.log(s) - np.log(s.shift(1)))
    df = df.dropna(subset=["log_ret"])

    ret_wide = df.pivot(index="date", columns="asset_id", values="log_ret").sort_index()

    n_total = ret_wide.shape[0]
    summary = pd.DataFrame({
        "n_obs": ret_wide.notna().sum(),
        "coverage": ret_wide.notna().sum() / n_total,
        "first_date": ret_wide.apply(lambda col: col.first_valid_index()),
        "last_date": ret_wide.apply(lambda col: col.last_valid_index()),
        "ann_vol": ret_wide.std() * np.sqrt(12),
    })

    summary["valid"] = (
        (summary["coverage"] >= coverage_min)
        & (summary["n_obs"] >= min_obs)
        & (summary["ann_vol"] >= min_vol)
    )

    clean_assets = summary.index[summary["valid"]]
    ret_clean = ret_wide[clean_assets].copy()

    print(f"{label}: {ret_wide.shape[1]} raw assets → {ret_clean.shape[1]} clean assets")
    print(f"{label}: {n_total} monthly observations; coverage threshold = {coverage_min:.0%}")

    return ret_wide, ret_clean, summary.sort_values("coverage", ascending=False)


indices_ret, indices_ret_clean, indices_summary = build_return_matrix(
    indices_raw,
    asset_cols=["tic"],
    label="Equity indices",
)

indices_ret_wide = indices_ret  # alias for downstream notebooks/exports


Equity indices: 735 raw assets → 469 clean assets
Equity indices: 249 monthly observations; coverage threshold = 80%


STEP 3B - Clean STOCKS data

In [34]:
stock_id_cols = ["gvkey"]
if "iid" in stocks_raw.columns:
    stock_id_cols.append("iid")

stocks_ret, stocks_ret_clean, stocks_summary = build_return_matrix(
    stocks_raw,
    asset_cols=stock_id_cols,
    label="Global stocks",
)

stocks_ret_wide = stocks_ret


Global stocks: 57672 raw assets → 21614 clean assets
Global stocks: 237 monthly observations; coverage threshold = 80%


STEP 3C - Align Dates & Merge into Multi-Asset Matrix

In [35]:
# Align dates across indices and stocks to build multi-asset matrices
common_dates_raw = indices_ret.index.intersection(stocks_ret.index)
multiasset_ret = pd.concat(
    [indices_ret.loc[common_dates_raw], stocks_ret.loc[common_dates_raw]],
    axis=1,
).sort_index()

common_dates_clean = indices_ret_clean.index.intersection(stocks_ret_clean.index)
multiasset_ret_clean = pd.concat(
    [indices_ret_clean.loc[common_dates_clean], stocks_ret_clean.loc[common_dates_clean]],
    axis=1,
).sort_index()

multiasset_ret = multiasset_ret.dropna(how="all")
multiasset_ret_clean = multiasset_ret_clean.dropna(how="all")

print("Raw multi-asset return matrix:", multiasset_ret.shape)
print("Clean multi-asset return matrix:", multiasset_ret_clean.shape)

multiasset_ret.head()


Raw multi-asset return matrix: (237, 58407)
Clean multi-asset return matrix: (237, 22083)


asset_id,86UNK080,I1EGY001,I1GHA001,I1ISR001,I1ISR002,I1JOR001,I1KEN001,I1MAR001,I1NGA001,I1QAT001,...,8169,8303,8544,8546,8594,9098,9135,9213,9501,9818
date,,,,,,,,,,,,,,,,,,,,,
2006-02-28,0.042366,-0.083322,0.007923,-0.025427,-0.029162,-0.092485,-0.027995,0.045030,0.006883,NaN,...,-0.016832,-0.364404,0.136201,-0.007187,0.103362,-0.142618,0.071801,-0.170626,0.033510,-0.306942
2006-03-31,-0.072036,-0.071837,0.007141,0.015154,0.017252,-0.066458,0.011034,0.054221,-0.021467,NaN,...,-0.014078,0.000000,-0.174954,0.015032,0.036952,0.042505,0.060489,-0.023530,0.021015,-0.015033
2006-04-30,0.076187,-0.024797,0.003372,0.051466,0.056100,-0.004278,-0.018810,0.094296,-0.001517,NaN,...,0.010577,0.144250,0.132261,-0.018464,0.057713,-0.019097,-0.017544,0.174353,-0.001076,0.048556
2006-05-31,-0.206216,-0.173879,0.013226,-0.012637,-0.010076,-0.017316,0.077541,-0.122189,0.060144,NaN,...,-0.033107,0.201295,0.048105,-0.052179,-0.084353,-0.027363,-0.033749,-0.020203,0.136667,0.147902
2006-06-30,-0.051347,-0.095593,-0.002164,-0.090652,-0.088866,-0.133334,-0.020734,0.003846,0.055625,NaN,...,-0.093861,0.093822,-0.072279,-0.002099,-0.003245,0.017027,0.027090,-0.107631,-0.040574,0.007679


In [36]:
def build_selection_report(summary_df, asset_class):
    df = summary_df.copy()
    df["asset_class"] = asset_class
    cols = ["asset_class", "coverage", "n_obs", "ann_vol", "first_date", "last_date", "valid"]
    return df[cols]

indices_selection = build_selection_report(indices_summary, "Index")
stocks_selection = build_selection_report(stocks_summary, "Stock")

selected_assets_report = pd.concat(
    [indices_selection.query("valid"), stocks_selection.query("valid")],
    axis=0,
).sort_values(["asset_class", "coverage"], ascending=[True, False])

print(f"Selected equity indices: {indices_selection['valid'].sum()}")
print(indices_selection.query("valid")[['coverage', 'n_obs', 'ann_vol']].head())

print(f"Selected stocks: {stocks_selection['valid'].sum()}")
print(stocks_selection.query("valid")[['coverage', 'n_obs', 'ann_vol']].head())

selected_assets_report.head()


Selected equity indices: 469
          coverage  n_obs   ann_vol
asset_id                           
I3GBR029       1.0    249  0.166417
I3IRL004       1.0    249  0.199049
I3NOR006       1.0    249  0.180726
I3NOR004       1.0    249  0.188800
I3NOR002       1.0    249  0.185945
Selected stocks: 21614
          coverage  n_obs   ann_vol
asset_id                           
100001         1.0    237  0.415220
242758         1.0    237  0.492659
242770         1.0    237  0.605144
242768         1.0    237  0.550517
242764         1.0    237  1.040357


,asset_class,coverage,n_obs,ann_vol,first_date,last_date,valid
asset_id,,,,,,,
I3GBR029,Index,1.0,249,0.166417,2005-02-28,2025-10-31,True
I3IRL004,Index,1.0,249,0.199049,2005-02-28,2025-10-31,True
I3NOR006,Index,1.0,249,0.180726,2005-02-28,2025-10-31,True
I3NOR004,Index,1.0,249,0.188800,2005-02-28,2025-10-31,True
I3NOR002,Index,1.0,249,0.185945,2005-02-28,2025-10-31,True


STEP 4 - Save Clean Data (for Streamlit and Track 2/3)

In [37]:
# STEP 4 - Save Clean Data (for Streamlit and Track 2/3)

indices_ret_clean.to_csv("clean_indices_returns.csv")
stocks_ret_clean.to_csv("clean_stocks_returns.csv")
multiasset_ret_clean.to_csv("clean_multiasset_returns.csv")

indices_summary.to_csv("indices_summary.csv")
stocks_summary.to_csv("stocks_summary.csv")
selected_assets_report.to_csv("selected_assets_report.csv")


Step 5 - Data-quality filtering

In [38]:
def qc_summary(summary_df, label, max_rows=5):
    kept = summary_df[summary_df["valid"]]
    dropped = summary_df[~summary_df["valid"]]

    print(f"{label}: kept {kept.shape[0]} assets, dropped {dropped.shape[0]} assets")
    if not dropped.empty:
        print(f"Top {max_rows} dropped assets (coverage, n_obs, ann_vol):")
        display_cols = dropped[["coverage", "n_obs", "ann_vol"]].head(max_rows)
        print(display_cols)
    return kept.head(max_rows)

qc_summary(indices_summary, "Equity indices")
qc_summary(stocks_summary, "Global stocks")

selected_assets_report.head(10)


Equity indices: kept 469 assets, dropped 266 assets
Top 5 dropped assets (coverage, n_obs, ann_vol):
          coverage  n_obs   ann_vol
asset_id                           
I3ITA029  0.787149    196  0.203576
I2CHN006  0.787149    196  0.229120
I6UNK154  0.779116    194  0.169942
I3GBR046  0.779116    194  0.178794
I3GBR031  0.779116    194  0.244983
Global stocks: kept 21614 assets, dropped 36058 assets
Top 5 dropped assets (coverage, n_obs, ann_vol):
          coverage  n_obs   ann_vol
asset_id                           
293741    0.797468    189  0.511483
293726    0.797468    189  0.467328
293795    0.797468    189  0.828561
293725    0.797468    189  0.448368
293796    0.797468    189  0.260623


,asset_class,coverage,n_obs,ann_vol,first_date,last_date,valid
asset_id,,,,,,,
I3GBR029,Index,1.0,249,0.166417,2005-02-28,2025-10-31,True
I3IRL004,Index,1.0,249,0.199049,2005-02-28,2025-10-31,True
I3NOR006,Index,1.0,249,0.180726,2005-02-28,2025-10-31,True
I3NOR004,Index,1.0,249,0.188800,2005-02-28,2025-10-31,True
I3NOR002,Index,1.0,249,0.185945,2005-02-28,2025-10-31,True
I3NLD014,Index,1.0,249,0.165208,2005-02-28,2025-10-31,True
I3NLD013,Index,1.0,249,0.176589,2005-02-28,2025-10-31,True
I3NLD012,Index,1.0,249,0.185657,2005-02-28,2025-10-31,True
I3MLT002,Index,1.0,249,0.127458,2005-02-28,2025-10-31,True


In [39]:
print("Indices (first 5 assets):")
print(indices_ret_clean.iloc[:5, :5])

print("Stocks (first 5 assets):")
print(stocks_ret_clean.iloc[:5, :5])

print("Multi-asset (first 5 assets):")
print(multiasset_ret_clean.iloc[:5, :5])


Indices (first 5 assets):
asset_id    I1ISR001  I1ISR002  I1MAR001  I1NGA001  I1SAU001
date                                                        
2005-02-28  0.021011  0.017429 -0.025755 -0.049771  0.099840
2005-03-31 -0.011563 -0.006082 -0.013714 -0.059645  0.143448
2005-04-30  0.017983  0.022529  0.020104  0.060018  0.094839
2005-05-31  0.032852  0.034076  0.046338 -0.022081  0.040401
2005-06-30 -0.063188 -0.061863 -0.003691  0.003842  0.112823
Stocks (first 5 assets):
asset_id      100001    100010    100012    100013    100022
date                                                        
2006-02-28  0.026545  0.077367  0.035156 -0.016823  0.080119
2006-03-31 -0.057577 -0.024031  0.018019  0.073796  0.123200
2006-04-30  0.041031  0.317878  0.007117  0.012973 -0.045513
2006-05-31 -0.080763 -0.132478 -0.128478 -0.084348 -0.060836
2006-06-30  0.031953  0.011510 -0.004040 -0.025467 -0.016459
Multi-asset (first 5 assets):
asset_id    I1ISR001  I1ISR002  I1MAR001  I1NGA001  I1SAU001
date